[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/04_activation_and_gating.ipynb)

# 04. Activation and gating — activation functions vs gated FFNs

ReLU, GELU, SiLU 같은 pointwise activation과 GLU/GEGLU/SwiGLU의 gated FFN 구조를 비교한다. Gated FFN은 같은 입력에서 두 learned projection을 만들고, gate activation과 elementwise product 뒤 output projection을 적용한다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. ReLU, GELU, and SiLU are pointwise nonlinearities

이 함수들은 입력 tensor의 각 원소를 독립적으로 변환한다. gated FFN과 달리 별도 branch interaction은 없다.


In [ ]:
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0], device=device)
print("ReLU:", F.relu(x))
print("GELU:", F.gelu(x))
print("SiLU:", F.silu(x))


## 2. Original GLU structure

GLU는 같은 input `x`에서 두 learned affine projection을 만들고 한쪽을 sigmoid gate로 사용한다. 핵심 연산은 `A(x) ⊙ sigmoid(B(x))`이다.


In [ ]:
batch = torch.randn(3, 8, device=device)
value_projection = nn.Linear(8, 16).to(device)
gate_projection = nn.Linear(8, 16).to(device)
value_branch = value_projection(batch)
gate_branch = torch.sigmoid(gate_projection(batch))
glu_hidden = value_branch * gate_branch
print("value branch:", value_branch.shape)
print("gate branch:", gate_branch.shape)
print("GLU hidden:", glu_hidden.shape)


## 3. GEGLU and SwiGLU as complete Transformer FFNs

Transformer의 gated FFN은 gate product 뒤 model dimension으로 다시 projection한다. GEGLU는 GELU gate를, SwiGLU는 SiLU gate를 사용한다.


In [ ]:
class GatedFFN(nn.Module):
    def __init__(self, model_dim=8, hidden_dim=24, gate="silu"):
        super().__init__()
        self.gate = gate
        self.value_projection = nn.Linear(model_dim, hidden_dim, bias=False)
        self.gate_projection = nn.Linear(model_dim, hidden_dim, bias=False)
        self.output_projection = nn.Linear(hidden_dim, model_dim, bias=False)

    def forward(self, x):
        value = self.value_projection(x)
        gate_input = self.gate_projection(x)
        if self.gate == "gelu":
            gate = F.gelu(gate_input)
        elif self.gate == "silu":
            gate = F.silu(gate_input)
        else:
            raise ValueError(self.gate)
        return self.output_projection(value * gate)

geglu_ffn = GatedFFN(gate="gelu").to(device)
swiglu_ffn = GatedFFN(gate="silu").to(device)
print("GEGLU output:", geglu_ffn(batch).shape)
print("SwiGLU output:", swiglu_ffn(batch).shape)


## 4. Compare ordinary FFN and SwiGLU computation graphs


In [ ]:
ordinary_ffn = nn.Sequential(
    nn.Linear(8, 24),
    nn.GELU(),
    nn.Linear(24, 8),
).to(device)
print("ordinary FFN:", ordinary_ffn(batch).shape)
print("SwiGLU FFN:", swiglu_ffn(batch).shape)


## References and provenance

**GELU** — Hendrycks & Gimpel.

**GLU** — Dauphin et al.

**GEGLU / SwiGLU** — Shazeer, *GLU Variants Improve Transformer*.
